In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/urdu-ocr-si26'
os.chdir(PROJECT_PATH)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/urdu-ocr-si26


## Week 2 — Image Preprocessing & Tesseract OCR Baseline

This notebook preprocesses the 101 Urdu images collected in Week 1
(grayscale, resize, denoise, binarize) to prepare them for model training.
It then tests Tesseract OCR on these images to evaluate how accurately
it handles Urdu text — which will help us understand why a custom model is needed.

In [ ]:
!pip install opencv-python-headless pillow --quiet

import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [ ]:
def preprocess_image(image_path, save_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    # Step 1: Convert to grayscale (removes colour noise)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize to standard size (keeps all images same dimensions)
    resized = cv2.resize(gray, (512, 128))

    # Step 3: Remove noise (makes text cleaner)
    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    # Step 4: Binarise (make pixels either pure black or pure white)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    # Save processed image
    cv2.imwrite(save_path, binary)
    return binary

# Create output folder
os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

Preprocessing function ready!


In [ ]:
import glob

# Find all images in data/raw/
all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)

print(f'Found {len(all_images)} images to process')

processed_count = 0
failed_images = []

for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    try:
        result = preprocess_image(img_path, save_path)
        if result is not None:
            processed_count += 1
        else:
            failed_images.append(img_path)
    except Exception as e:
        failed_images.append(img_path)
        print(f'Error on {img_path}: {e}')

print(f'\nDone! Processed {processed_count} images')
print(f'Failed: {len(failed_images)} images')
if failed_images:
    print('Failed files:', failed_images)
print('Check data/processed/ folder')

Found 101 images to process

Done! Processed 101 images
Failed: 0 images
Check data/processed/ folder


In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd --quiet
!pip install pytesseract --quiet

import pytesseract
from PIL import Image

# Test on 5 of your processed images
test_images = list(glob.glob('data/processed/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()

for img_path in test_images:
    img = Image.open(img_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 1s (1,040 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
=== Tesseract Results on Urdu Images ===

Image: data/processed/کلو میٹر.png
Tesseract output: 
---
Image: data/processed/تجارتی جہاز.p

## Gap Analysis — Tesseract OCR on Urdu Images

**Image 1: کلو میٹر.png**
- Actual text: کلو میٹر
- Tesseract output: (empty — nothing detected)
- What went wrong: Complete failure to detect any text.

**Image 2: تجارتی جہاز.png**
- Actual text: تجارتی جہاز
- Tesseract output: تجادخی چنہاز
- What went wrong: Wrong characters — letters were misread and jumbled, producing a completely different (meaningless) word.

**Image 3: سسٹم میں خرابی.png**
- Actual text: سسٹم میں خرابی
- Tesseract output: (empty — nothing detected)
- What went wrong: Complete failure to detect any text.

**Image 4: عبوری حکومت.png**
- Actual text: عبوری حکومت
- Tesseract output: سبوری حکیم
- What went wrong: Wrong characters — similar-looking letters were confused (ع→س, ت→م), resulting in a different, incorrect word.

**Image 5: نماز جنازہ.png**
- Actual text: نماز جنازہ
- Tesseract output: (empty — nothing detected)
- What went wrong: Complete failure to detect any text.

**Summary:**
Tesseract fails on Urdu because the script is cursive and context-dependent — letters change shape depending on their position in a word (beginning, middle, end), which confuses Tesseract's character segmentation. It also struggles with the right-to-left flow and closely-spaced joined letters, often either detecting nothing at all or misreading visually similar letters (like ع and س) as one another. This shows that a generic OCR engine trained mainly on Latin-script and limited Urdu data cannot reliably read real-world Urdu text — which is exactly why we need a custom-trained model for this project.

In [8]:
import shutil

# processed folder copy karein
if os.path.exists('/content/repo_upload/data/processed'):
    shutil.rmtree('/content/repo_upload/data/processed')
shutil.copytree('/content/drive/MyDrive/urdu-ocr-si26/data/processed', '/content/repo_upload/data/processed')

print("Processed folder copy ho gaya")

os.chdir('/content/repo_upload')
os.system('git config user.email "warood@example.com"')
os.system('git config user.name "waroodzahrakhan"')
os.system('git add data/processed')
os.system('git commit -m "Add Week 2 preprocessed images"')
result = os.system('git push')
print("Push status:", "Success" if result == 0 else "Failed")

Processed folder copy ho gaya
Push status: Success


In [9]:
import shutil, glob

# Colab notebook Drive mein kahan save hoti hai, usay dhoondein
possible_paths = glob.glob('/content/drive/MyDrive/**/SI26-Week2-Warood.ipynb', recursive=True)
possible_paths += glob.glob('/content/drive/MyDrive/**/SI26_Week2_Warood.ipynb', recursive=True)

print("Mili file(s):", possible_paths)

Mili file(s): ['/content/drive/MyDrive/Colab Notebooks/SI26-Week2-Warood.ipynb']


In [10]:
import shutil, os

notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week2-Warood.ipynb'
notebook_dest = '/content/repo_upload/SI26-Week2-Warood.ipynb'

shutil.copy(notebook_source, notebook_dest)
print("Notebook copy ho gaya")

os.chdir('/content/repo_upload')
os.system('git add SI26-Week2-Warood.ipynb')
os.system('git commit -m "Add Week 2 notebook"')
result = os.system('git push')
print("Push status:", "Success" if result == 0 else "Failed")

Notebook copy ho gaya
Push status: Failed


In [11]:
import subprocess

os.chdir('/content/repo_upload')

result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

STDOUT: 
STDERR: remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: 0a0bf53ae3700f6d01fd1bf2f6735ef97626c746        
remote:            path: SI26-Week2-Warood.ipynb:1        
remote:             
remote:        (?) To push, remove secret from commit(s)

In [13]:
import shutil, os

notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week2-Warood.ipynb'
notebook_dest = '/content/repo_upload/SI26-Week2-Warood.ipynb'

shutil.copy(notebook_source, notebook_dest)
print("Notebook copy ho gaya")

os.chdir('/content/repo_upload')
os.system('git add SI26-Week2-Warood.ipynb')
os.system('git commit -m "Add Week 2 notebook"')
result = os.system('git push')
print("Push status:", "Success" if result == 0 else "Failed")

Notebook copy ho gaya
Push status: Success


In [14]:
os.chdir('/content/repo_upload')
os.system('git log --oneline -5')

0

In [15]:
import subprocess

os.chdir('/content/repo_upload')
result = subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True)
print(result.stdout)

1c863e4 Add Week 2 preprocessed images
23eaa03 Update README.md
b42808b Remove duplicate loose images from root
956eea8 Add complete labeled dataset
5f70602 Rename تجارتی جہاز.png to data/raw/newspaper/تجارتی جہاز.png



In [19]:
push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)
print("PUSH RETURN CODE:", push_result.returncode)

PUSH STDOUT: 
PUSH STDERR: Everything up-to-date

PUSH RETURN CODE: 0


In [20]:
import subprocess, os

os.chdir('/content/repo_upload')

# Identity set karein
subprocess.run(['git', 'config', 'user.email', 'warood@example.com'])
subprocess.run(['git', 'config', 'user.name', 'waroodzahrakhan'])

# Check karein notebook file add hui thi ya nahi
status_result = subprocess.run(['git', 'status'], capture_output=True, text=True)
print("STATUS:", status_result.stdout)

STATUS: On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   SI26-Week2-Warood.ipynb




In [21]:
commit_result = subprocess.run(['git', 'commit', '-m', 'Add Week 2 notebook'], capture_output=True, text=True)
print("COMMIT:", commit_result.stdout, commit_result.stderr)

push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)
print("PUSH RETURN CODE:", push_result.returncode)

COMMIT: [main a17c20b] Add Week 2 notebook
 1 file changed, 1 insertion(+)
 create mode 100644 SI26-Week2-Warood.ipynb
 
PUSH STDOUT: 
PUSH STDERR: remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: a17c20b5983f2980b6780fb652610df5cf160a98        
remot

In [22]:
import subprocess, shutil, os

notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week2-Warood.ipynb'
notebook_dest = '/content/repo_upload/SI26-Week2-Warood.ipynb'

shutil.copy(notebook_source, notebook_dest)
print("Notebook copy ho gaya (token-free version)")

os.chdir('/content/repo_upload')
subprocess.run(['git', 'add', 'SI26-Week2-Warood.ipynb'])

commit_result = subprocess.run(['git', 'commit', '-m', 'Add Week 2 notebook (clean)'], capture_output=True, text=True)
print("COMMIT:", commit_result.stdout, commit_result.stderr)

push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)
print("PUSH RETURN CODE:", push_result.returncode)

Notebook copy ho gaya (token-free version)
COMMIT: [main a923811] Add Week 2 notebook (clean)
 1 file changed, 1 insertion(+), 1 deletion(-)
 
PUSH STDOUT: 
PUSH STDERR: remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: a923811dd1ceb6f263ddc49e0705fe4c

In [23]:
from getpass import getpass
import subprocess, os

TOKEN = getpass("GitHub token paste karein aur Enter dabayein: ")

os.chdir('/content/repo_upload')
new_remote_url = f"https://{TOKEN}@github.com/waroodzahrakhan/urdu-ocr-codesaviours-si26-warood.git"
subprocess.run(['git', 'remote', 'set-url', 'origin', new_remote_url])
print("Remote set ho gaya, token kahin save nahi hua")

GitHub token paste karein aur Enter dabayein: ··········
Remote set ho gaya, token kahin save nahi hua


In [24]:
import shutil

notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week2-Warood.ipynb'
notebook_dest = '/content/repo_upload/SI26-Week2-Warood.ipynb'
shutil.copy(notebook_source, notebook_dest)

subprocess.run(['git', 'add', 'SI26-Week2-Warood.ipynb'])
commit_result = subprocess.run(['git', 'commit', '-m', 'Add Week 2 notebook (clean)'], capture_output=True, text=True)
print("COMMIT:", commit_result.stdout, commit_result.stderr)

push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)
print("PUSH RETURN CODE:", push_result.returncode)

COMMIT: [main e1cb18e] Add Week 2 notebook (clean)
 1 file changed, 1 insertion(+), 1 deletion(-)
 
PUSH STDOUT: 
PUSH STDERR: remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: a923811dd1ceb6f263ddc49e0705fe4c109a3b17        
remote:            path: S

In [25]:
import subprocess, os

os.chdir('/content/repo_upload')

reset_result = subprocess.run(['git', 'reset', '--hard', 'origin/main'], capture_output=True, text=True)
print("RESET:", reset_result.stdout, reset_result.stderr)


RESET: HEAD is now at 1c863e4 Add Week 2 preprocessed images
 
